# IRA PDF Form Fill

## form1065_filler.py
==================
Python class that fills an IRS Form 1065 PDF (U.S. Return of Partnership Income)
from a plain Python dict.

## Supports two fill modes
-----------------------
1. FILLABLE  – The official IRS PDF already has AcroForm fields (most versions do).
               The class detects field names and writes values directly.
2. OVERLAY   – Falls back to ReportLab coordinate-based text overlay when a field
               is not found in the AcroForm, or when given a flat/scanned PDF.

## Quick start
-----------
    from form1065_filler import Form1065Filler, build_1065_dict_from_gl

    # Build from GL (uses the Form1065Preparer output dict)
    data = build_1065_dict_from_gl(gl_dict, entity_name="Sunset Ridge Rentals LLC",
                                   ein="12-3456789", tax_year=2024)

    filler = Form1065Filler("f1065.pdf")          # official IRS PDF
    filler.fill(data)
    filler.save("Form_1065_Filled.pdf")

    # Or use the all-in-one method (also creates a blank template):
    Form1065Filler.fill_from_dict("f1065.pdf", data, "Form_1065_Filled.pdf")


Dependencies
------------
````
  pip install pypdf reportlab
````

In [1]:
# Import services
from __future__ import annotations

import io
import os
import re
import copy
from typing import Any, Dict, List, Optional, Tuple

# ── pypdf ────────────────────────────────────────────────────────────────────
try:
    from pypdf import PdfReader, PdfWriter
    from pypdf.generic import (
        NameObject, create_string_object, ArrayObject,
        DictionaryObject, BooleanObject, NumberObject,
        IndirectObject,
    )
    _PYPDF = True
except ImportError:
    _PYPDF = False

# ── reportlab (overlay fallback) ────────────────────────────────────────────
try:
    from reportlab.pdfgen import canvas as rl_canvas
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.units import inch
    _REPORTLAB = True
except ImportError:
    _REPORTLAB = False

print(f"IMPORTS: : _PYPDF={_PYPDF}, _REPORTLAB: ={_REPORTLAB}")

IMPORTS: : _PYPDF=True, _REPORTLAB: =True


In [15]:
# Init GL and import form 1065 data
import json
with open("_JUNK_FILL_1065.json", 'r') as fio:
    data = json.load(fio)

print("Form 1065 Imported into data")

# General Ledger - All accounts
import os
from pathlib import Path

# LLC General Ledger
from ledger.LLC import LLC

top = os.path.join(Path.home(), 'GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group')
llc = LLC('WBGroupLLC',debug=False, top=top)
# Save entity information
eDict = llc.entity
acctDIR = os.path.join(llc.TOP, llc.dirAccounting, str(llc.yr))
yeDIR = os.path.join(acctDIR, 'YE_Tax Records')


Form 1065 Imported into data


In [4]:
#  Class pdfCore : .init / .print_field_map /._auto_compute / .fill_from_dict / .inspect_fields
# ════════════════════════════════════════════════════════════════════════════

class pdfCore:
    """
    Fill IRS Form 1065 PDF fields from a Python dict.

    Parameters
    ----------
    pdf_path : str
        Path to the *blank* IRS Form 1065 PDF (downloadable from irs.gov).
    use_cents : bool
        If True, format dollar amounts with two decimal places.
    verbose : bool
        Print field-fill status to stdout.
    """

    def __init__(
        self,
        pdf_path: str,
        use_cents: bool = False,
        verbose:   bool = True,
        debug : bool = False,
        acro : Bool = True, # Acro first, False : ignore acro, do only overlay
        form : Any = None
    ):
        if not _PYPDF:
            raise ImportError("pypdf is required:  pip install pypdf")
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"PDF not found: {pdf_path}")

        self.pdf_path  = pdf_path
        self.use_cents = use_cents
        self.verbose   = verbose
        self.debug = debug
        self.acro = acro
        self.form = form

        self._reader: PdfReader   = PdfReader(pdf_path)
        self._writer: PdfWriter   = PdfWriter()
        self._writer.append(self._reader)   # clone all pages


    def save(self, output_path: str) -> str:
        """Write filled PDF to *output_path*. Returns the path."""
        # flatten form so values are visible in all readers
        if hasattr(self._writer, "_root_object"):
            try:
                # Mark NeedAppearances so readers regenerate field appearances
                acroform = self._writer._root_object.get("/AcroForm")
                if acroform:
                    acroform.update({
                        NameObject("/NeedAppearances"): BooleanObject(True)
                    })
            except Exception:
                pass

        with open(output_path, "wb") as f:
            self._writer.write(f)
        if self.verbose:
            print(f"\n  💾  Saved → {output_path}")
        return output_path

    # ── auto-compute totals ──────────────────────────────────────────────────

    @staticmethod
    def _auto_compute(data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Fill in computed lines that the caller omitted.
        All computations mirror IRS Form 1065 arithmetic.
        """
        def get(k): return float(data.get(k) or 0)

        # Line 1c = 1a − 1b
        if "line_1c" not in data:
            v = get("line_1a") - get("line_1b")
            if v: data["line_1c"] = v

        # Line 8  = total income
        if "line_8" not in data:
            total = (get("line_1c") + get("line_3") + get("line_4") +
                     get("line_5")  + get("line_6") + get("line_7"))
            if total: data["line_8"] = total

        # Line 16c = 16a − 16b
        if "line_16c" not in data:
            v = get("line_16a") - get("line_16b")
            if v: data["line_16c"] = v

        # Line 21 = total deductions
        if "line_21" not in data:
            total = sum(get(f"line_{n}") for n in
                        [9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20]) + get("line_16c")
            if total: data["line_21"] = total

        # Line 22 = ordinary income/(loss)
        if "line_22" not in data:
            v = get("line_8") - get("line_21")
            data["line_22"] = v

        # Schedule K line 1 mirrors line 22
        if "k_line_1" not in data and "line_22" in data:
            data["k_line_1"] = data["line_22"]

        # Schedule M-2 ending capital
        if "m2_end_capital" not in data:
            v = (get("m2_beg_capital") + get("m2_capital_contrib") +
                 get("m2_net_income")  - get("m2_distributions"))
            if v: data["m2_end_capital"] = v

        return data

    # ── class-method convenience ─────────────────────────────────────────────

    @classmethod
    def fill_from_dict(
        cls,
        input_pdf:   str,
        data:        Dict[str, Any],
        output_pdf:  str,
        verbose:     bool = True,
    ) -> str:
        """One-liner: load blank PDF, fill, save. Returns output path."""
        filler = cls(input_pdf, verbose=verbose)
        filler.fill(data)
        return filler.save(output_pdf)

    @classmethod
    def inspect_fields(cls, pdf_path: str) -> Dict[str, Any]:
        """
        Return a dict of all AcroForm fields found in the PDF.
        Useful for debugging / mapping new form versions.
        """
        reader = PdfReader(pdf_path)
        fields = {}
        raw = reader.get_fields()
        if raw:
            for name, field in raw.items():
                fields[name] = {
                    "type":  field.get("/FT"),
                    "value": field.get("/V"),
                    "rect":  field.get("/Rect"),
                }
        return fields

    @classmethod
    def print_field_map(cls) -> None:
        """Print the built-in FIELD_MAP in a readable table."""
        print(f"\n{'Key':<25} {'AcroForm Field':<48} {'Pg':>3}  {'x':>4} {'y':>4}")
        print("─" * 88)
        for key, (af, pg, x, y, *_) in FIELD_MAP.items():
            af_short = (af or "OVERLAY")[-46:]
            print(f"  {key:<23} {af_short:<48} {pg:>3}  {x:>4.0f} {y:>4.0f}")

    def _to_text(self, key: str, value: Any) -> str:
        """Convert a dict value to the string to write into the PDF field."""
        if key in BOOL_FIELDS:
            return CHECKBOX_ON if value else CHECKBOX_OFF
        if isinstance(value, bool):
            return "X" if value else ""
        if isinstance(value, (int, float)):
            return (
                _fmt_dollar_cents(value) if self.use_cents
                else _fmt_dollar(value)
            )
        return str(value)


In [5]:
# class Form1065Fill: .fill / .save 
'''
.fill(dataDict)
'''
class Form1065Fill(pdfCore):

    # ── public API ───────────────────────────────────────────────────────────

    def fill(self, data: Dict[str, Any]) -> "Form1065Filler":
        if self.verbose : print(f"----- {self.__class__.__name__} START : DataLen:{len(data)}" )
        if self.acro : 
            pf = pdfAcro(self.pdf_path, form=self)
            return pf.fill(data)

        pf = pdfOverlay(self.pdf_path, form=self)
        return pf.fill(data)
            

    def fillAll(self, data: Dict[str, Any]) -> "Form1065Filler":
        """
        Fill fields from *data* dict. Applies auto-computations for
        totals if the caller omitted them.
        Returns self for chaining.
        """
        data = self._auto_compute(dict(data))
        filled_acro    = 0
        filled_overlay = 0
        overlay_ops: Dict[int, List] = {}   # page → list of (x,y,w,h,text,fs)

        for key, value in data.items():
            if key not in FIELD_MAP:
                if self.verbose:
                    print(f"  [SKIP]    '{key}' not in FIELD_MAP")
                continue

            acro_name, page, x, y, w, h, fs = FIELD_MAP[key]
            text_val = self._to_text(key, value)

            # ── try AcroForm first ────────────────────────────────────────
            acro_ok = False 
            if self.acro and acro_name:
                acro_ok = self._fill_acro(acro_name, f"a_{text_val}", key in BOOL_FIELDS, value)
                if acro_ok:
                    filled_acro += 1
                    if self.verbose:
                        print(f"  [ACRO]    {key:30s} = {text_val}")

            # ── fallback overlay ─────────────────────────────────────────
            if not acro_ok:
                if not _REPORTLAB:
                    if self.verbose:
                        print(f"  [SKIP-OVL] {key} — reportlab not installed")
                    continue
                overlay_ops.setdefault(page, []).append((x, y, w, h, f"o_{text_val}", fs))
                filled_overlay += 1
                if self.verbose:
                    print(f"  [OVERLAY] {key:30s} = {text_val}  (p{page})")

        # apply overlays page by page
        if overlay_ops:
            self._apply_overlays(overlay_ops)

        if self.verbose:
            print(f"\n  ✅  Filled {filled_acro} AcroForm fields + "
                  f"{filled_overlay} overlay fields")
        return self

    def _apply_overlays(self, ops: Dict[int, List]) -> None:
        """Render text overlays using ReportLab, then merge into writer pages."""
        from reportlab.pdfgen import canvas as rl_canvas
        from pypdf import PdfReader as _R

        page_h = 792.0   # letter height in points

        for page_num, fields in ops.items():
            buf = io.BytesIO()
            c   = rl_canvas.Canvas(buf, pagesize=(612, page_h))
            c.setFont("Helvetica", 9)

            for (x, y, w, h, text, fs) in fields:
                c.setFont("Helvetica", fs)
                # right-align numbers, left-align text
                is_num = bool(re.match(r"^[\d,.()\-]+$", text.replace(" ", "")))
                if is_num:
                    c.drawRightString(x + w - 2, y + 2, text)
                else:
                    c.drawString(x + 2, y + 2, text)

            c.save()
            buf.seek(0)
            overlay_reader = _R(buf)
            overlay_page   = overlay_reader.pages[0]

            # merge overlay onto the target writer page (0-indexed)
            target = self._writer.pages[page_num - 1]
            target.merge_page(overlay_page)

    def _fill_acro(
        self,
        acro_name: str,
        text_val:  str,
        is_bool:   bool,
        raw_value: Any,
    ) -> bool:
        """
        Write a value into an AcroForm field.
        Returns True if the field was found and written.
        """
        # pypdf 3+ uses update_page_form_field_values
        # We iterate writer pages and their annotations directly for reliability.
        found = False
        for page in self._writer.pages:
            if "/Annots" not in page:
                continue
            annots = page["/Annots"]
            for annot_ref in annots:
                try:
                    annot = annot_ref.get_object() if hasattr(annot_ref, "get_object") else annot_ref
                except Exception:
                    continue
                if annot.get("/T") == acro_name or str(annot.get("/T")) == acro_name:
                    # determine value to write
                    if is_bool:
                        write_val = CHECKBOX_ON if raw_value else CHECKBOX_OFF
                        annot.update({NameObject("/V"): NameObject(write_val),
                                       NameObject("/AS"): NameObject(write_val)})
                    else:
                        annot.update({NameObject("/V"): create_string_object(text_val)})
                    found = True

        # Also try pypdf's high-level API as a second pass
        if not found:
            try:
                self._writer.update_page_form_field_values(
                    self._writer.pages[0] if len(self._writer.pages) == 1 else None,
                    {acro_name: CHECKBOX_ON if (is_bool and raw_value) else
                                CHECKBOX_OFF if (is_bool and not raw_value) else
                                text_val},
                )
                found = True
            except Exception:
                pass

        return found

In [6]:
# class pdfAcro(pdfCore): ._fill_acro 
class pdfAcro(pdfCore):

    def __init__(self, pdf_path, **kwargs):
        super().__init__(pdf_path, **kwargs)
        # discover actual AcroForm field names in the PDF
        self._acro_fields: Dict[str, Any] = self._discover_acro_fields()
        if self.verbose:
            print(f"[Form1065Filler] Loaded '{pdf_path}'  "
                  f"({len(self._reader.pages)} pages, "
                  f"{len(self._acro_fields)} AcroForm fields detected)")


    # ── internal helpers ─────────────────────────────────────────────────────

    def _discover_acro_fields(self) -> Dict[str, Any]:
        """Return dict of {field_name: field_obj} from the PDF's AcroForm."""
        raw = self._reader.get_fields()
        if not raw:
            return {}
        return {name: obj for name, obj in raw.items()}

    def fill(self, data: Dict[str, Any]) -> "Form1065Filler":
        """
        Fill fields from *data* dict. Applies auto-computations for
        totals if the caller omitted them.
        Returns self for chaining.
        """
        if self.verbose : print(f"----- {self.__class__.__name__} START : DataLen:{len(data)}" )
        data = self._auto_compute(dict(data))
        filled_acro    = 0

        for key, value in data.items():
            if key not in FIELD_MAP:
                if self.verbose:
                    print(f"  [SKIP]    '{key}' not in FIELD_MAP")
                continue

            acro_name, page, x, y, w, h, fs = FIELD_MAP[key]
            text_val = self._to_text(key, value)

            # ── try AcroForm first ────────────────────────────────────────
            if not acro_name: continue
            acro_ok = self._fill_acro(acro_name, f"a_{text_val}", key in BOOL_FIELDS, value)
            if acro_ok:
                filled_acro += 1
                if self.verbose:
                    print(f"  [ACRO]    {key:30s} = {text_val}")

        if self.verbose:
            print(f"\n  ✅  Filled {filled_acro} AcroForm fields + "
                  f"{filled_overlay} overlay fields")
        return self
    

    def _fill_acro(
        self,
        acro_name: str,
        text_val:  str,
        is_bool:   bool,
        raw_value: Any,
    ) -> bool:
        """
        Write a value into an AcroForm field.
        Returns True if the field was found and written.
        """
        # pypdf 3+ uses update_page_form_field_values
        # We iterate writer pages and their annotations directly for reliability.
        found = False
        for page in self._writer.pages:
            if "/Annots" not in page:
                continue
            annots = page["/Annots"]
            for annot_ref in annots:
                try:
                    annot = annot_ref.get_object() if hasattr(annot_ref, "get_object") else annot_ref
                except Exception:
                    continue
                if annot.get("/T") == acro_name or str(annot.get("/T")) == acro_name:
                    # determine value to write
                    if is_bool:
                        write_val = CHECKBOX_ON if raw_value else CHECKBOX_OFF
                        annot.update({NameObject("/V"): NameObject(write_val),
                                       NameObject("/AS"): NameObject(write_val)})
                    else:
                        annot.update({NameObject("/V"): create_string_object(text_val)})
                    found = True

        # Also try pypdf's high-level API as a second pass
        if not found:
            try:
                self._writer.update_page_form_field_values(
                    self._writer.pages[0] if len(self._writer.pages) == 1 else None,
                    {acro_name: CHECKBOX_ON if (is_bool and raw_value) else
                                CHECKBOX_OFF if (is_bool and not raw_value) else
                                text_val},
                )
                found = True
            except Exception:
                pass

        return found

In [7]:
# class pdfOverlay(pdfCore): .apply_overlays / ._to_text / ._auto_compute
class pdfOverlay(pdfCore):

    def __init__(self, pdf_path, **kwargs):
        super().__init__(pdf_path, **kwargs)

    def fill(self, data: Dict[str, Any]) -> "Form1065Filler":
        """
        Fill fields from *data* dict. Applies auto-computations for
        totals if the caller omitted them.
        Returns self for chaining.
        """
        if self.verbose : print(f"----- {self.__class__.__name__} START : DataLen:{len(data)}" )
        data = self._auto_compute(dict(data))
        filled_overlay = 0
        overlay_ops: Dict[int, List] = {}   # page → list of (x,y,w,h,text,fs)

        for key, value in data.items():
            if key not in FIELD_MAP:
                if self.verbose:
                    print(f"  [SKIP]    '{key}' not in FIELD_MAP")
                continue

            acro_name, page, x, y, w, h, fs = FIELD_MAP[key]
            text_val = self._to_text(key, value)

            # ── fallback overlay ─────────────────────────────────────────
            if not _REPORTLAB:
                if self.verbose:
                    print(f"  [SKIP-OVL] {key} — reportlab not installed")
                continue
            overlay_ops.setdefault(page, []).append((x, y, w, h, f"o_{text_val}", fs))
            filled_overlay += 1
            if self.verbose:
                print(f"  [OVERLAY] {key:30s} = {text_val}  (p{page})")

        # apply overlays page by page
        if overlay_ops:
            self._apply_overlays(overlay_ops)

        if self.verbose:
            print(f"\n  ✅  Filled {filled_overlay} / {len(data)} overlay fields")
        return self






In [12]:
# PDF Test Form 1065
dTestDict = {k:k for k in FIELD_MAP.keys()}
dTestDict
# ── Attempt to fill PDF if it exists ────────────────────────────────────

PDF_IN  = os.path.join(yeDIR, "Form_1065-IRS.pdf")
PDF_OUT = os.path.join("junk_Form_1065_TEST.pdf")

# Fill the PDF
filler = Form1065Fill(PDF_IN, verbose=False, acro=False)
filler.fillAll(dTestDict)
filler.save(PDF_OUT)

'junk_Form_1065_TEST.pdf'

In [9]:
# ── Attempt to fill PDF if it exists ────────────────────────────────────
yeDIR = os.path.join(llc.TOP, llc.dirAccounting, str(llc.yr), 'YE_Tax Records')

PDF_IN  = os.path.join(yeDIR, "Form_1065-IRS.pdf")
PDF_OUT = os.path.join(yeDIR, "Form_1065_Filled.pdf")


if False:
    # Fill the PDF
    filler = Form1065Filler(PDF_IN, verbose=True)
    filler.fill(data)
    filler.save(PDF_OUT)
    print(f"\nDone. Open '{PDF_OUT}' to review.")

In [13]:
fDict = filler.inspect_fields(PDF_IN)
fDict

{'topmostSubform[0]': {'type': None, 'value': None, 'rect': None},
 'topmostSubform[0].Page1[0]': {'type': None, 'value': None, 'rect': None},
 'topmostSubform[0].Page1[0].HeaderAddress_ReadOrder[0]': {'type': None,
  'value': None,
  'rect': None},
 'topmostSubform[0].Page1[0].HeaderAddress_ReadOrder[0].CalendarName_ReadOrder[0]': {'type': None,
  'value': None,
  'rect': None},
 'topmostSubform[0].Page1[0].HeaderAddress_ReadOrder[0].CalendarName_ReadOrder[0].f1_01[0]': {'type': '/Tx',
  'value': None,
  'rect': None},
 'topmostSubform[0].Page1[0].HeaderAddress_ReadOrder[0].CalendarName_ReadOrder[0].f1_02[0]': {'type': '/Tx',
  'value': None,
  'rect': None},
 'topmostSubform[0].Page1[0].HeaderAddress_ReadOrder[0].CalendarName_ReadOrder[0].f1_03[0]': {'type': '/Tx',
  'value': None,
  'rect': None},
 'topmostSubform[0].Page1[0].HeaderAddress_ReadOrder[0].CalendarName_ReadOrder[0].f1_04[0]': {'type': '/Tx',
  'value': None,
  'rect': None},
 'topmostSubform[0].Page1[0].HeaderAddress_Re

In [28]:
dict1 = {'a': 1, 'b': 2, 'c' : 0}
dict2 = {'b': 3, 'c': 4}
dict1 | dict2

{'a': 1, 'b': 3, 'c': 4}

In [10]:
# build 1065
# ════════════════════════════════════════════════════════════════════════════
#  CONVENIENCE BUILDER  –  GL dict  →  1065 data dict
#  (bridges Form1065Preparer output to Form1065Filler input)
# ════════════════════════════════════════════════════════════════════════════

def build_1065_dict_from_gl(
    gl: Dict[str, float],
    entity_name:   str  = "LLC Rental Partnership",
    ein:           str  = "XX-XXXXXXX",
    tax_year:      int  = 2024,
    address:       str  = "",
    city_state_zip:str  = "",
    business_code: str  = "531110",   # IRS code: Lessors of residential buildings
    preparer_name: str  = "",
    preparer_ptin: str  = "",
    preparer_date: str  = "",
) -> Dict[str, Any]:
    """
    Convert a raw General Ledger dict (same format used by Form1065Preparer)
    directly into a Form1065Filler data dict.

    GL sign convention:
      Positive = money IN  (income, capital contributions, ending balance)
      Negative = money OUT (expenses, asset purchases)
    """
    def pos(k):   return max(0.0, float(gl.get(k, 0)))
    def absv(k):  return abs(float(gl.get(k, 0)))

    # income
    line_1a  = pos("Acct.Cash.Income")
    line_5_k = pos("Acct.Interest.Income")
    line_7   = pos("Acct.Cash.Misc")
    line_8   = line_1a + line_5_k + line_7

    # deductions
    line_20  = absv("Acct.Cash.Expense") + absv("Acct.Cash.Util")
    line_21  = line_20
    line_22  = line_8 - line_21

    # balance sheet
    fixed    = absv("Acct.Asset.Purchase")
    cash_end = pos("Balance")
    cap_cont = pos("Acct.Cash.Investment")

    return {
        # header
        "entity_name":        entity_name,
        "ein":                ein,
        "tax_year":           tax_year,
        "address":            address,
        "city_state_zip":     city_state_zip,
        "principal_product":  "Residential Rental Property",
        "business_code":      business_code,
        "number_of_k1s":      1,

        # page 1 income
        "line_1a":  line_1a,
        "line_7":   line_7,
        "line_8":   line_8,

        # page 1 deductions
        "line_20":  line_20,
        "line_21":  line_21,
        "line_22":  line_22,

        # schedule K
        "k_line_1":  line_22,
        "k_line_2":  line_1a,
        "k_line_5":  line_5_k,
        "k_line_11": line_7,

        # schedule L
        "l_cash_beg":         0.0,
        "l_cash_end":         cash_end,
        "l_fixed_assets_beg": 0.0,
        "l_fixed_assets_end": fixed,
        "l_total_assets_beg": 0.0,
        "l_total_assets_end": cash_end + fixed,
        "l_partners_cap_beg": 0.0,
        "l_partners_cap_end": cap_cont + line_22,
        "l_total_liab_beg":   0.0,
        "l_total_liab_end":   cap_cont + line_22,

        # schedule M-2
        "m2_beg_capital":    0.0,
        "m2_capital_contrib": cap_cont,
        "m2_net_income":      line_22,
        "m2_end_capital":     cap_cont + line_22,

        # preparer block
        "preparer_name": preparer_name,
        "preparer_ptin": preparer_ptin,
        "preparer_date": preparer_date,
    }

In [11]:
# ════════════════════════════════════════════════════════════════════════════
#  DEMO
# ════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import json, sys

    # ── General Ledger (from previous session) ───────────────────────────────
    GL = {
        "Acct.Asset.Purchase":  -214_113.95,
        "Acct.Cash.Expense":      -1_766.92,
        "Acct.Cash.Income":        4_000.53,
        "Acct.Cash.Investment":  219_227.00,
        "Acct.Cash.Misc":             29.47,
        "Acct.Cash.Util":         -1_056.95,
        "Acct.Interest.Income":      400.00,
        "Balance":                 6_719.18,
    }

    # ── Build 1065 data dict ─────────────────────────────────────────────────
    data = build_1065_dict_from_gl(
        GL,
        entity_name    = "Sunset Ridge Rentals LLC",
        ein            = "12-3456789",
        tax_year       = 2024,
        address        = "123 Hilltop Drive",
        city_state_zip = "Austin, TX 78701",
        preparer_name  = "Jane Smith CPA",
        preparer_ptin  = "P01234567",
        preparer_date  = "04/15/2025",
    )

    print("\n── 1065 Data Dict (JSON) ──")
    print(json.dumps({k: v for k, v in data.items()
                      if not isinstance(v, bool)}, indent=2))

    # ── Attempt to fill PDF if it exists ────────────────────────────────────
    PDF_IN  = "f1065.pdf"
    PDF_OUT = "Form_1065_Filled.pdf"

    if not os.path.exists(PDF_IN):
        print(f"\n[INFO] '{PDF_IN}' not found locally.")
        print("  Download the blank form from:")
        print("  https://www.irs.gov/pub/irs-pdf/f1065.pdf")
        print("  Then re-run:  python form1065_filler.py")
        print("\n  Field map preview:")
        Form1065Filler.print_field_map()
        sys.exit(0)

    # Fill the PDF
    filler = Form1065Filler(PDF_IN, verbose=True)
    filler.fill(data)
    filler.save(PDF_OUT)
    print(f"\nDone. Open '{PDF_OUT}' to review.")


── 1065 Data Dict (JSON) ──
{
  "entity_name": "Sunset Ridge Rentals LLC",
  "ein": "12-3456789",
  "tax_year": 2024,
  "address": "123 Hilltop Drive",
  "city_state_zip": "Austin, TX 78701",
  "principal_product": "Residential Rental Property",
  "business_code": "531110",
  "number_of_k1s": 1,
  "line_1a": 4000.53,
  "line_7": 29.47,
  "line_8": 4430.000000000001,
  "line_20": 2823.87,
  "line_21": 2823.87,
  "line_22": 1606.130000000001,
  "k_line_1": 1606.130000000001,
  "k_line_2": 4000.53,
  "k_line_5": 400.0,
  "k_line_11": 29.47,
  "l_cash_beg": 0.0,
  "l_cash_end": 6719.18,
  "l_fixed_assets_beg": 0.0,
  "l_fixed_assets_end": 214113.95,
  "l_total_assets_beg": 0.0,
  "l_total_assets_end": 220833.13,
  "l_partners_cap_beg": 0.0,
  "l_partners_cap_end": 220833.13,
  "l_total_liab_beg": 0.0,
  "l_total_liab_end": 220833.13,
  "m2_beg_capital": 0.0,
  "m2_capital_contrib": 219227.0,
  "m2_net_income": 1606.130000000001,
  "m2_end_capital": 220833.13,
  "preparer_name": "Jane Smit

NameError: name 'Form1065Filler' is not defined

## Example of Find/Fill fields

Found first field: 'f1_01[0]'. Attempting to fill it.


AttributeError: 'PdfWriter' object has no attribute 'update_page_form_fields'